# Cohort creation on UK-Biobank
... talk about whats going to happen, how we crate cohort.

**Note:**
The notebooks used to create the sample-QCED datasets are based on or inspired by the GWAS AD Guide for the UK Biobank (
[GWAS_AD_Tutorial](https://dnanexus.gitbook.io/uk-biobank-rap/science-corner/gwas-ex)).
This guide focuses on creating an AD-By-Proxy dataset and to perform early sample QC-steps.
The second part of the original tutorial, which covers GWAS settings and performing GWAS, is not included here.

## Initial steps
First of all, we need to extract the data that we are working with. 
Please take a look at the [create_initial.ipynb](../create_initial.ipynb) notebook to check the steps performed to create the data-csv. 

In [ ]:
import ast
import re
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_palette('tab10')

In [ ]:
# This dataframe was created separately. Check the corresponding notebook (create_initial.ipynb) for this. 
df_all = pd.read_csv("data/df_from_sql_new.csv", delimiter="\t", header=0)
df_all.rename(columns=lambda x: re.sub('participant.','',x), inplace=True)

In [ ]:
# Read in ids that have genotype data available
id_file="data/all_geno_snps.ids"
ids_df=pd.read_csv(id_file,names=['eid'], dtype={'eid': str})
df = df_all.astype({'eid':str}).merge(ids_df, on='eid', how='inner')

## Derive Phenotypes
### AD Cases

In [ ]:
ad_case_score=4

In [ ]:
# Please refer to the data coding file created eartly in the create_initial.ipynb notebook. 
codings_df = pd.read_csv('data_codings.csv')

# Collapse ICD-10 codes for Alzheimer's disease (G30* and F00*)
ad_icd_codes = list(
    codings_df[(codings_df["coding_name"] == "data_coding_19") & ((codings_df["parent_code"] == "G30") | (codings_df["parent_code"] == "F00"))]["code"])
ad_icd_codes

In [ ]:
# Replace NaN with string None for p41270 -> ICD10 diagnoses
# Note: eval will return Nonetype for string "None"
df.loc[:,"p41270"] = df["p41270"].replace(np.nan, "None")

In [ ]:
# Get each participant's hospital inpatient records in ICD10 Diagnoses
def icd10_codes(row):
	icd10_codes = ast.literal_eval(row['p41270']) or []
	return list(set(icd10_codes))

df['icd10_codes'] = df.apply(icd10_codes, axis=1)

def has_ad_icd10(row):
# If the participant has any of the ICD-10 codes for AD, record the risk to ad_case_score 
	return 0 if set(row['icd10_codes']).isdisjoint(ad_icd_codes) else ad_case_score 

df['has_ad_icd10'] = df.apply(has_ad_icd10, axis=1)

In [ ]:
# Check overall #NUM ad-patients
df['has_ad_icd10'].value_counts()

### AD Proxy
Check parental information

In [ ]:
# Now, let's check the parents illness records
def get_illness_records(row):
    combined = []
    # some arguments are/where formatted as strings. 
    # Convert them back to a list
    for col in row:
        if isinstance(col, list):
            combined.extend(col)
        elif isinstance(col, str):
            combined.extend(ast.literal_eval(col))
    return combined

In [ ]:
df['illnesses_of_mother'] = df.filter(regex=('p20110*')).apply(get_illness_records , axis=1)
df['illnesses_of_father'] = df.filter(regex=('p20107*')).apply(get_illness_records, axis=1)

In [ ]:
# Get the max age between age at death and recorded age
df['father_age'] = df.filter(regex=(r'(p1807_*|p2946_*)')).max(axis=1)
df['mother_age'] = df.filter(regex=(r'(p3526_*|p1845_*)')).max(axis=1)
# If the parent has diagnosed with AD (code 10: https://biobank.ndph.ox.ac.uk/showcase/coding.cgi?id=1010), record risk as 1; 
# else assign parent's AD risk with their risk, which is their age (proportional to diff of age of 100) with minimum risk at 0.32 

def parents_ad_risk(row): 
	# cap risk of healthy individuals at 0.32
	m_age=min(100, row['mother_age'])
	f_age=min(100, row['father_age'])
	father_ad_risk = 1 if 10 in row['illnesses_of_father'] else np.minimum(0.32, (100 - f_age)/100)
	mother_ad_risk = 1 if 10 in row['illnesses_of_mother'] else np.minimum(0.32, (100 - m_age)/100)
	return father_ad_risk + mother_ad_risk 

#df['parents_ad_risk_old'] = df.apply(parents_ad_risk_old, axis=1)
df['parents_ad_risk'] = df.apply(parents_ad_risk, axis=1)
df['ad_risk_by_proxy'] = df[['has_ad_icd10', 'parents_ad_risk']].max(axis=1) 

In [ ]:
# Sanity -> Check affected siblings
# This is not included in the ad_by_proxy risk calculation.
#  None of the controls have affected siblings anyways. 
def has_affected_sibling(row):
    return 1 if 10 in row['illnesses_of_sibling'] else 0

df['illnesses_of_sibling'] = df.filter(regex=('p20111*')).apply(get_illness_records, axis=1)
df['has_affected_sibling'] = df.apply(has_affected_sibling, axis=1)

print(f'total affected siblings:\n', df['has_affected_sibling'].value_counts())
print(f'affected siblings of controls:\n', df[df['ad_risk_by_proxy'] < 1.0]['has_affected_sibling'].value_counts())

sns.histplot(df, x='ad_risk_by_proxy', hue='has_affected_sibling', bins=30, multiple='stack')

In [ ]:
def ad_proxy_score(row):
    r = row['ad_risk_by_proxy']
    if r >= 2:
        return r
    if r > 0.64:
        return 1
    return 0
df['ad_by_proxy'] = df.apply(ad_proxy_score, axis=1)

In [ ]:
def get_ad_parents(row):
	father_ad = 1 if 10 in row['illnesses_of_father'] else 0
	mother_ad = 1 if 10 in row['illnesses_of_mother'] else 0
	return father_ad + mother_ad
    
df['num_ad_parent']=df.apply(get_ad_parents, axis=1)
df['num_ad_parent'].value_counts()

In [ ]:
def get_status(row):
    r = row['ad_risk_by_proxy']
    if r == 4:
        return 'case'
    if (r > 0.64) & (r <= 2.0):
        return 'proxy'
    return 'control'
df['status'] = df.apply(get_status, axis=1)

In [ ]:
# Also match with STR data from PopSTR
idx_file="data/str_sample_ids"
str_ids=pd.read_csv(idx_file,names=['eid'], dtype={'eid': str})
str_ids.shape[0]
df['in_str_data'] = np.where(df['eid'].isin(str_ids['eid']), 1, 0)
df['in_str_data'].value_counts()

## Early QC
We want to do some early QC do filter out patients that we are sure about to remove.  
These are some really early sample-QC steps we need before we can say anything about the actual data distribution. 

In [ ]:
# get individual numers of reduced
mis_sex = df[df['p31'] != df['p22001']] # Filter in sex and genetic sex are the same
aneuploidy = df[df['p22019']==1]       # Not Sex chromosome aneuploidy
het_outlier = df[df['p22027']==1]       # Not heterozygosity or missingnes outliers

# This is the number of samples we have in our table 
print(f'samples in sex mismatch\nn={mis_sex["status"].value_counts()}\ntotal={mis_sex.shape[0]}')
print(f'\nsamples in aneuploidy\nn={aneuploidy["status"].value_counts()}\ntotal={aneuploidy.shape[0]}')
print(f'\n samples in outlier for heterozygosity or missingness \nn={het_outlier["status"].value_counts()}\ntotal={het_outlier.shape[0]}')

In [ ]:
# there are probably overlaps between removed samples
combined = aneuploidy['eid'].to_list() + mis_sex['eid'].to_list() + het_outlier['eid'].to_list() 
print(len(combined))
print(len(set(combined)))

In [ ]:
# now actually filter them
df_filter1 = df[
			(df['p31'] == df['p22001']) & # Filter in sex and genetic sex are the same
			(df['p22019'].isnull()) &       # Not Sex chromosome aneuploidy
			(df['p22027'].isnull())       # Not heterozygosity or missingnes outliers
]
# This is the number of samples we have in our table 
lost=df.shape[0]-df_filter1.shape[0]
print(f'reducing by {lost} to {df_filter1.shape[0]} samples')

In [ ]:
# be safe
df_c = df.copy(deep=True)
df = pd.DataFrame()
df.shape[0], df_c.shape[0]

## Early QC II - Filter parents
In contrast to the tutorial, we will (for now) not remove 'unreliable' patients from cases since we do not need a proxy-phenotype for diagnosed cases anyways
Also, detailed information from diagnosed AD-Patients is less reliable as well. 

After an early QC, we keep all AD patients from this set.  
For deriving the by-proxy risk/people, we need to do some extra QC regarding the reliability of their parent's information.

But first, we do an extra sanity check of how many AD-people wee would loose if we would do this for them as well.

In [ ]:
# This is only for checking
ad_df=df_filter1[df_filter1['has_ad_icd10'] == ad_case_score]
print("Cases before QC", ad_df.shape[0])
check_df=ad_df[
	# is there father's and mother's age
	((ad_df['father_age'].notnull()) & (ad_df['father_age']>0)) &  
	((ad_df['mother_age'].notnull()) & (ad_df['mother_age']>0)) & 
	# Filter out "do not know" or "prefer not to answer" parents illnesses for group 1 and group 2
	(ad_df['illnesses_of_father'].apply(lambda x:(-11 not in x and -13 not in x and -21 not in x and -23 not in x))) & 
	(ad_df['illnesses_of_mother'].apply(lambda x:(-11 not in x and -13 not in x and -21 not in x and -23 not in x)))   
]

print(f'after QC:{check_df.shape[0]}; potentially removed: {ad_df.shape[0]-check_df.shape[0]}')

We only remove controls or potential proxys where information about parents is missing

In [ ]:
ctr_df = df_filter1[df_filter1['has_ad_icd10'] == 0]
print(f'controls/proxys before: {ctr_df.shape[0]}')
age_qced = ctr_df[
    # is there father's and mother's age
    ((ctr_df['father_age'].notnull()) & (ctr_df['father_age']>0)) &  
    ((ctr_df['mother_age'].notnull()) & (ctr_df['mother_age']>0)) &  
    # Filter out "do not know" or "prefer not to answer" parents illnesses for group 1 and group 2
    (ctr_df['illnesses_of_father'].apply(lambda x:(-11 not in x and -13 not in x and -21 not in x and -23 not in x))) & 
    (ctr_df['illnesses_of_mother'].apply(lambda x:(-11 not in x and -13 not in x and -21 not in x and -23 not in x)))   
]

print(f'after QC:{age_qced.shape[0]}; actually removed: {ctr_df.shape[0]-age_qced.shape[0]}')

In [ ]:
before_ids=set(ctr_df['eid'].to_list())
print(len(before_ids))
after_ids=set(age_qced['eid'].to_list())
print(len(after_ids))
age_rmed_ids = before_ids.difference(after_ids)
print(len(age_rmed_ids))
excessives=age_qced[age_qced.p22021 == 10]['eid'].to_list()
excluded=age_qced[age_qced.p22021 == -1]['eid'].to_list()
print(len(excessives))
print(len(excluded))

age_rmed_ids.update(set(excessives))
age_rmed_ids.update(set(excluded))
print(len(age_rmed_ids))

pd.DataFrame(age_rmed_ids).to_csv('cohort_data/kin_ad_extraremove.csv', index=False)


We now combine cases and parental-qced controls

In [ ]:
# Now, lets combine them back again
df_qced = pd.concat([ad_df, age_qced])
total=df_qced.shape[0]
print("total: ", total)
print(df_qced['status'].value_counts())
print(df_qced['has_ad_icd10'].value_counts())

### QC III - remove excessive kinships


In [ ]:
print(df_qced.p22021.value_counts())
df_qced.drop(df_qced[df_qced.p22021 == 10].index, inplace=True)
df_qced.drop(df_qced[df_qced.p22021 == -1].index, inplace=True)
df_qced.p22021.value_counts()

In [ ]:
# Now, lets combine them back again
total=df_qced.shape[0]
print("total: ", total)
print(df_qced['status'].value_counts())
print(df_qced['has_ad_icd10'].value_counts())

## Ethnic Grouping

In [ ]:
t = df_qced.filter(regex=(r'p21000_*')).fillna(0)
t.iloc[0]

In [ ]:
df_qced['reported_ethnicity_i0'] = df_qced['p21000_i0'].fillna(value='-5.0')
#df_qced['reported_ethnicity_i0'].value_counts()

In [ ]:
e_replace={
    r'^\-[135].0':'no_answ',
    '1001.0':'white_british',
    r'(^1.0$)|(1002.0)|(1003.0)':'other_white',
    r'^2[01234]{0,3}\.0':'mixed', 
    r'^3[01234]{0,3}\.0':'asian', 
    r'^4[01234]{0,3}\.0':'black', 
    '5.0':'chinese', 
    '6.0':'other'
}
df_qced['ethnic_group']=df_qced['reported_ethnicity_i0'].astype('str').replace(e_replace, regex=True)

In [ ]:
df_qced.ethnic_group.value_counts()

In [ ]:
def filter_cols(df):
    df = df.rename(columns=lambda x: re.sub('p22009_a','pc',x)) 
    df = df.rename(columns={'eid':'IID', 'p31': 'sex', 'p21022': 'age', 'p22006': 'genetic_group', 
    'p22019': 'sex_chromosome_aneuploidy', 
    'p22021': 'kinship_to_other_participants', 
    'p22027': 'outliers_for_heterozygosity_or_missing'})
    
    # Add FID column -- required input format for plink 
    df['#FID'] = df['IID'] 

    # Create a phenotype table from our QCed data 
    return df[[
        '#FID', 'IID', 'sex', 'age',
        'status', 
        'ad_by_proxy', 'ad_risk_by_proxy', 
        'ethnic_group',
        'reported_ethnicity_i0',
        'kinship_to_other_participants',
        'has_ad_icd10', 
        ] + [f'pc{i}' for i in range(1, 41)]].reset_index(drop=True)

In [ ]:
df_sub = filter_cols(df_qced)

In [ ]:
def save_pheno(df, file_path):
    # Create a phenotype table from our QCed data 
    df_fin = df[['#FID', 'IID', 'sex', 
        'ad_risk_by_proxy',
        'status',
        'ethnic_group',
        'reported_ethnicity_i0',
        'ad_by_proxy',
        'kinship_to_other_participants',
        'has_ad_icd10',
        ]
    ]
    
    df_sex = df_fin.replace({'sex': 0}, value='2')
    df_sex.to_csv(file_path, sep='\t', na_rep='-9', index=False, quoting=3)
    return df_sex

In [ ]:
all_qced = save_pheno(df_sub, "all_qced.phe")

# Export samples with kinships

In [ ]:
relateds=all_qced[all_qced['kinship_to_other_participants']==1]
save_pheno(relateds, 'related_samples.phe')['ad_by_proxy'].value_counts()

... seperate processing of kinships using plinks king algorithm

In [ ]:
rm_relative_ids=pd.read_csv('relatives_third.king.cutoff.out.id3_AD', sep='\t', dtype='str')
drop_df = all_qced[all_qced['#FID'].astype(str).isin(rm_relative_ids['#FID'])]
# for sanity
all_qced['kinship_rm'] = np.where(all_qced['#FID'].astype(str).isin(rm_relative_ids['#FID']), 1, 0)

In [ ]:
str_df = all_qced.drop(drop_df.index)
save_pheno(str_df, 'data/cohort_data/str_data.phe')

In [ ]:
str_df['kinship_rm'].value_counts() 

In [ ]:
for k in str_df.ethnic_group.unique():
    save_pheno(str_df[str_df.ethnic_group == k], f'data/cohort_data/{k}.phe')